# CMAPSS FD001 — Modeling, Evaluation, and SHAP Interpretation

This notebook trains and evaluates the RUL prediction model and produces the SHAP analysis that connects model output to physical turbofan degradation. It is the final step in the pipeline before the trained artifact is handed to the FastAPI backend.

**What this notebook produces:**
- `models/xgb_rul.joblib` — the trained XGBoost model loaded by `api/predictor.py`
- RMSE scores for both the Ridge baseline and XGBoost on the official FD001 test set
- A SHAP feature importance ranking tied back to known HPC degradation mechanisms

**Modelling philosophy:** The goal is not to squeeze out the lowest possible RMSE through exhaustive hyperparameter search. The goal is a clean, interpretable, well-documented model that edges out the Zheng et al. 2017 deep LSTM benchmark (RMSE 16.14) and clears the Babu et al. 2016 CNN (18.45) using only tabular features and a gradient boosting model — no recurrent networks, no sequence padding, no look-ahead. The SHAP analysis is as important as the RMSE: it demonstrates that the model has learned physically meaningful patterns, not statistical artefacts.

---
**Environment:** `conda activate cmapss-rul` from the project root.

In [ ]:
from pathlib import Path
import sys, joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW    = ROOT / "data" / "raw"
MODELS = ROOT / "models"
sys.path.insert(0, str(ROOT))

from src.features import load_raw, add_rul, build_features, RUL_CLIP
from src.model import (get_feature_cols, train_test_split_by_unit,
                       train_xgb, evaluate)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
print("Imports OK.")

## 1. Load and Engineer Features

We apply the same `build_features()` pipeline from `02_feature_engineering.ipynb`. No new transforms — feature engineering is frozen in `src/features.py`. The training set gets RUL labels via `add_rul()`; the test set does not (true labels come from `RUL_FD001.txt`).

In [ ]:
train_raw    = load_raw(RAW / "train_FD001.txt")
train_labeled = add_rul(train_raw.copy())
train_fe     = build_features(train_labeled)

test_raw  = load_raw(RAW / "test_FD001.txt")
test_fe   = build_features(test_raw)
rul_true  = np.clip(np.loadtxt(RAW / "RUL_FD001.txt"), 0, RUL_CLIP)

feature_cols = get_feature_cols(train_fe)
last_rows    = test_fe.groupby("unit").last().reset_index()
X_test       = last_rows[feature_cols]
y_test       = rul_true

print(f"Training rows:    {len(train_fe):,}")
print(f"Training engines: {train_fe['unit'].nunique()}")
print(f"Feature columns:  {len(feature_cols)}")
print(f"Test engines:     {len(y_test)}")

## 2. Grouped Train / Validation Split

We hold out 20 of the 100 training engines as a validation set. The split is strictly by engine unit — `train_test_split_by_unit()` ensures that no cycle from a held-out engine appears in training. This is the only correct strategy for time-series sensor data: a random row-level split would place early and late cycles from the same engine in both splits, leaking per-engine drift information and producing misleadingly low error estimates.

The official FD001 test set (100 separate engines) is held completely aside until final evaluation. Validation RMSE guides model development; test RMSE is the number that goes in the README.

In [ ]:
rng = np.random.default_rng(42)
all_units = train_fe["unit"].unique()
val_units = rng.choice(all_units, size=20, replace=False).tolist()

train_split, val_split = train_test_split_by_unit(train_fe, val_units)

X_tr = train_split[feature_cols];  y_tr = train_split["rul"]
X_va = val_split[feature_cols];    y_va = val_split["rul"]

print(f"Train split: {len(X_tr):,} rows — {train_fe['unit'].nunique() - len(val_units)} engines")
print(f"Val split:   {len(X_va):,} rows — {len(val_units)} engines")
print(f"Val units: {sorted(val_units)}")

## 3. Baseline — Ridge Regression

Before training XGBoost, we fit a Ridge regression baseline. This serves two purposes. First, it establishes a minimum bar: any model that doesn't beat Ridge on the same features has a problem. Second, it helps interpret the XGBoost improvement — we can attribute the gap in RMSE specifically to the non-linearity and interaction terms that gradient boosting captures but linear regression cannot.

Ridge requires standardised features (unlike XGBoost), so we fit a `StandardScaler` on the training split only and apply it to validation and test.

In [ ]:
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_va_sc = scaler.transform(X_va)

ridge = Ridge(alpha=1.0)
ridge.fit(X_tr_sc, y_tr)

lr_val_preds  = np.clip(ridge.predict(X_va_sc), 0, 125)
lr_val_rmse   = np.sqrt(mean_squared_error(y_va, lr_val_preds))

lr_test_preds = np.clip(ridge.predict(scaler.transform(X_test)), 0, 125)
lr_test_rmse  = np.sqrt(mean_squared_error(y_test, lr_test_preds))

print(f"Ridge RMSE — validation split: {lr_val_rmse:.3f}")
print(f"Ridge RMSE — official test set: {lr_test_rmse:.3f}")

## 4. XGBoost — Primary Model

XGBoost is the right choice for this problem for three concrete reasons:

1. **Non-linear degradation.** Sensor-RUL relationships are non-linear — temperature    rise accelerates near failure, not linearly. XGBoost captures this naturally;    linear regression requires manual polynomial expansion.

2. **Feature interactions.** Multiple sensors degrade together (HPC temperature rises    *and* coolant bleed increases *and* pressure drops). XGBoost learns these    cross-sensor interactions through tree splits; linear models treat each feature    independently.

3. **SHAP compatibility.** XGBoost's tree structure admits exact SHAP values via    `booster.predict(pred_contribs=True)` — each feature's contribution to a    specific prediction is computed in polynomial time with no approximation.

The hyperparameters below were chosen to prevent overfitting on 80 training engines: moderate depth (6), low learning rate (0.05), and 80% row/column subsampling.

In [ ]:
xgb_model = train_xgb(X_tr, y_tr)

xgb_val   = evaluate(xgb_model, X_va, y_va)
xgb_test_preds = np.clip(xgb_model.predict(X_test), 0, 125)
xgb_test_rmse  = np.sqrt(mean_squared_error(y_test, xgb_test_preds))

print(f"XGBoost RMSE — validation split: {xgb_val['rmse']:.3f}")
print(f"XGBoost RMSE — official test set: {xgb_test_rmse:.3f}")
print(f"Zheng et al. 2017 deep LSTM:       16.14   (Babu et al. 2016 CNN: 18.45)")
print()
improvement = lr_test_rmse - xgb_test_rmse
print(f"XGBoost improvement over Ridge:  {improvement:.3f} RMSE cycles")
print(f"XGBoost vs. Zheng LSTM (16.14):   {16.14 - xgb_test_rmse:.3f} RMSE cycles (positive = better)")
print("Note: recent transformer/hybrid SOTA on FD001 is ~11-14 — do NOT claim SOTA.")

## 5. Results Summary

| Model | Val RMSE | Test RMSE (FD001) | Notes |
|-------|----------|-------------------|-------|
| Ridge Regression | — | — | Baseline |
| **XGBoost** | **—** | **—** | Primary model |
| Zheng et al. 2017 — deep LSTM | — | 16.14 | Published benchmark |
| Babu et al. 2016 — CNN | — | 18.45 | Earlier CNN baseline |

*(Fill in the dashes with the values printed above — they match what appears in the README.)*

The predicted vs. actual scatter plot below shows the error structure on the test set. Colour distinguishes *early predictions* (blue: predicted RUL lower than true — safe, triggers unnecessary maintenance) from *late predictions* (red: predicted RUL higher than true — the operationally dangerous case). A well-calibrated model has more blue than red and keeps late errors small in magnitude.

In [ ]:
errors = xgb_test_preds - y_test
late   = errors > 0
early  = ~late

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test[early], xgb_test_preds[early], alpha=0.7, s=50,
           color="#2563eb", label="Early prediction (safe)", edgecolors="white", linewidths=0.4)
ax.scatter(y_test[late],  xgb_test_preds[late],  alpha=0.7, s=50,
           color="#ef4444", label="Late prediction (riskier)", edgecolors="white", linewidths=0.4)
ax.plot([0, 125], [0, 125], "k--", linewidth=1.2, alpha=0.5, label="Perfect prediction")
ax.set_xlabel("True RUL (cycles)", fontsize=11)
ax.set_ylabel("Predicted RUL (cycles)", fontsize=11)
ax.set_title(f"Predicted vs. True RUL — FD001 Test Set\nRMSE = {xgb_test_rmse:.2f} cycles",
             fontsize=11)
ax.legend(fontsize=9)
ax.set_xlim(-3, 128); ax.set_ylim(-3, 128)
plt.tight_layout()
plt.show()

late_count  = late.sum()
early_count = early.sum()
print(f"Early predictions (safe):    {early_count} / 100")
print(f"Late predictions (riskier):  {late_count}  / 100")
print(f"Max late error:              {errors[late].max():.1f} cycles")

## 6. SHAP Feature Attribution — What the Model Has Learned

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions. A positive SHAP value for a feature means it pushed the prediction toward a *longer* remaining life; negative means it pushed toward *shorter*.

We use `booster.predict(pred_contribs=True)` — XGBoost's native exact SHAP implementation — which avoids the version-compatibility issues of the `shap.TreeExplainer` wrapper while producing identical values.

The beeswarm plot below shows 300 training-set predictions. Each dot is one row (one engine-cycle); its horizontal position is the SHAP value (in cycle units) and its colour is the feature value (red = high, blue = low). Reading the plot left to right for `sensor_3_mean30`: high feature values (red dots) push predictions to the left — shorter remaining life. That is the expected signature of HPC outlet temperature: a hotter compressor outlet means more fouling, less remaining life.

In [ ]:
# Use XGBoost's built-in exact SHAP (compatible with XGBoost 3.x)
sample_idx  = rng.choice(len(train_fe), size=300, replace=False)
X_shap_df   = train_fe.iloc[sample_idx][feature_cols].reset_index(drop=True)
d_shap      = xgb.DMatrix(X_shap_df, feature_names=feature_cols)
shap_matrix = xgb_model.get_booster().predict(d_shap, pred_contribs=True)[:, :-1]

mean_abs    = np.abs(shap_matrix).mean(axis=0)
top15_idx   = np.argsort(mean_abs)[::-1][:15]
top15_cols  = [feature_cols[i] for i in top15_idx]
sv_top15    = shap_matrix[:, top15_idx]
X_top15     = X_shap_df[top15_cols]

print("Top 10 features by mean |SHAP|:")
for i in np.argsort(mean_abs)[::-1][:10]:
    print(f"  {feature_cols[i]:35s}  {mean_abs[i]:.3f} cycles")

In [ ]:
fig = plt.figure(figsize=(9, 6.5))
shap.summary_plot(sv_top15, X_top15, show=False, plot_size=None)
plt.gca().set_title("SHAP Values — Top 15 Features (training sample, n=300)",
                    fontsize=11, pad=10)
plt.tight_layout()
plt.show()

## 7. Physical Interpretation of SHAP Results

This is what separates a model that *works* from a model you can *explain* — and it's what the CMAPSS literature calls the "prognostic reasoning" layer.

**`sensor_3_mean30` — HPC outlet temperature, 30-cycle rolling mean**  
The dominant feature by a wide margin. Sensor 3 measures the temperature at the outlet of the High-Pressure Compressor. As the HPC fouls over time (deposits on compressor blades reduce airfoil efficiency), the compressor must work harder to achieve the same pressure ratio, which raises outlet temperature. A rising 30-cycle mean in sensor 3 is the primary thermodynamic signature of HPC degradation — the exact fault mode simulated in FD001.

**`sensor_2_mean30` — LPC outlet temperature, 30-cycle rolling mean**  
The LPC (Low-Pressure Compressor) feeds air into the HPC. As HPC efficiency drops, the aerodynamic loading on the LPC changes, which is reflected in its outlet temperature. Sensor 2 rising together with sensor 3 is a correlated signature of the same fault propagating upstream.

**`sensor_11` and `sensor_11_std30` — HPC outlet coolant bleed**  
Coolant bleed from the HPC is used to cool the turbine blades downstream. As the HPC degrades, the bleed flow characteristics change — both the absolute value and its cycle-to-cycle variability (`_std30`) increase. The model has learned to use both the level and the *noisiness* of this channel as degradation indicators.

**`sensor_9_mean30` — bypass ratio**  
The bypass ratio (the fraction of total airflow that bypasses the core) shifts as compressor efficiency changes. A declining bypass ratio mean signals that more air is being forced through the degraded core, consistent with advancing HPC fouling.

**`sensor_14_mean30` and `sensor_14_std30` — LPT outlet coolant bleed**  
Similar to sensor 11: the LPT (Low-Pressure Turbine) coolant bleed reflects thermal stress downstream of the degraded HPC. Both the level and variability of this channel carry late-stage degradation signal.

The pattern across all top features is consistent: the model has identified the temperature and flow sensors that directly respond to HPC fouling, and it uses their rolling statistics — not their instantaneous values — to estimate how far along the degradation curve the engine is. That is a physically meaningful learned representation, not a statistical coincidence.

In [ ]:
joblib.dump(xgb_model, MODELS / "xgb_rul.joblib")
print(f"Model saved → {MODELS / 'xgb_rul.joblib'}")
print(f"File size:  {(MODELS / 'xgb_rul.joblib').stat().st_size / 1024:.1f} KB")
print()
print("── Final Results ─────────────────────────────────────────────")
print(f"Ridge RMSE  (official test set): {lr_test_rmse:.3f}")
print(f"XGBoost RMSE (official test set): {xgb_test_rmse:.3f}")
print(f"Zheng et al. 2017 deep LSTM:      16.14   (Babu 2016 CNN: 18.45)")
print()
print("Model artifact is ready for api/predictor.py to load at server startup.")

## Next Steps — Deployment

The model artifact at `models/xgb_rul.joblib` is everything the API needs. The FastAPI backend (`api/main.py`) loads it once at server startup via `api/predictor.py:load_model()` and reuses it for every request.

To test the full stack locally:

```bash
uvicorn api.main:app --reload
# Visit http://localhost:8000/docs
# Upload frontend/sample_engine.csv via the frontend at http://127.0.0.1:5500
```

To deploy:
1. Push this repo (including `models/xgb_rul.joblib`) to GitHub.
2. Connect to Render via the `render.yaml` Blueprint — one click.
3. Deploy frontend to Vercel with root directory set to `frontend/`.
4. Update the three URL placeholders: `API_BASE` in `app.js`,    `allow_origins` in `api/main.py`, and the Live Demo links in `README.md`.